# CRISP-DM Phase 4: Feature Selection
## SelectKBest (chi2, k=7) — Crédit Nationale Azur
This notebook performs feature selection on the cleaned dataset to identify
the 7 most statistically significant predictors of loan acceptance.

The chi-squared (chi2) scoring function measures the statistical independence
between each feature and the target variable. Features with low chi2 scores
are more likely to be irrelevant and are candidates for removal.

CRITICAL PIPELINE ORDER:
1. Load cleaned data
2. Split into train/test (stratified)
3. One-hot encode categorical variables
4. Apply SelectKBest(chi2) — BEFORE StandardScaler
5. Identify and document exactly which feature was dropped
6. Results feed directly into notebooks 05 and 06

WHY chi2 must come before StandardScaler:
StandardScaler introduces negative values. The chi-squared test requires
all input values to be non-negative. Applying scaling first would
mathematically invalidate the chi2 test.

In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2

# Load cleaned dataset
df = joblib.load('../data/cleaned_data.pkl')

# Verify no NaNs
print(f"NaN values in loaded pkl: {df.isnull().sum().sum()}")
print(f"Dataset shape: {df.shape}")
print(f"\nTarget column unique values: {df['personal_loan'].unique()}")

# Encode target
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})

print(f"\nNaN in target after mapping: {df['personal_loan'].isnull().sum()}")
print(f"Target value counts:\n{df['personal_loan'].value_counts()}")

X = df.drop('personal_loan', axis=1)
y = df['personal_loan']

print(f"\nNaN values in X: {X.isnull().sum().sum()}")
print(f"Features: {list(X.columns)}")

NaN values in loaded pkl: 0
Dataset shape: (6000, 12)

Target column unique values: <StringArray>
['no', 'yes']
Length: 2, dtype: str

NaN in target after mapping: 0
Target value counts:
personal_loan
0    5100
1     900
Name: count, dtype: int64

NaN values in X: 0
Features: ['age', 'yrs_experience', 'family_size', 'education_level', 'income', 'mortgage_amt', 'credit_card_acct', 'credit_card_spend', 'share_trading_acct', 'fixed_deposit_acct', 'online_acct']


## Step 1: Stratified Train/Test Split
Stratification preserves the 85/15 class ratio in both sets,
preventing algorithm bias from the imbalanced target variable.
.copy() is applied to avoid Pandas SettingWithCopyWarning
during subsequent transformations.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test  = X_test.copy()

print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")
print(f"Train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Test class balance:  {y_test.value_counts(normalize=True).round(3).to_dict()}")

Train shape: (4800, 11) | Test shape: (1200, 11)
Train class balance: {0: 0.85, 1: 0.15}
Test class balance:  {0: 0.85, 1: 0.15}


## Step 2: One-Hot Encode Categorical Variables
Categorical variables are encoded post-split to prevent data leakage.
Each column is encoded individually and concatenated back, retaining
column headers — following the Pandas approach recommended in this course.
The test set is aligned to the training set's columns after encoding
to handle any categories present in one set but not the other.

In [4]:
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']

for col in categorical_cols:
    # Encode training set
    dummies_train = pd.get_dummies(X_train[col], prefix=col)
    X_train.drop([col], axis=1, inplace=True)
    X_train = pd.concat([X_train, dummies_train], axis=1)

    # Encode test set
    dummies_test = pd.get_dummies(X_test[col], prefix=col)
    X_test.drop([col], axis=1, inplace=True)
    X_test = pd.concat([X_test, dummies_test], axis=1)

# Align test columns to training columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"Feature count after encoding: {X_train.shape[1]}")
print(f"All columns:\n{list(X_train.columns)}")

Feature count after encoding: 15
All columns:
['age', 'yrs_experience', 'family_size', 'income', 'mortgage_amt', 'credit_card_spend', 'share_trading_acct', 'fixed_deposit_acct', 'education_level_Advanced or Professional', 'education_level_Graduate', 'education_level_Undergraduate', 'credit_card_acct_no', 'credit_card_acct_yes', 'online_acct_no', 'online_acct_yes']


## Step 3: Apply SelectKBest(chi2, k=7)
SelectKBest scores each feature's statistical independence from the target.
k=7 retains the 7 highest-scoring features.

IMPORTANT: SelectKBest outputs a headless NumPy array. The selected feature
names must be manually cross-referenced against the original DataFrame columns
to identify exactly which features were retained and which were dropped.

This step is performed on training data only. The same selector is then
applied (transform only, no re-fit) to the test set.

In [5]:
selector = SelectKBest(chi2, k=7)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected  = selector.transform(X_test)

# Cross-reference headless array output against original column names
selected_mask     = selector.get_support()
selected_features = X_train.columns[selected_mask].tolist()
dropped_features  = X_train.columns[~selected_mask].tolist()

print(f"Total features before selection: {X_train.shape[1]}")
print(f"\nRetained features ({len(selected_features)}):")
for f in selected_features:
    print(f"  + {f}")
print(f"\nDropped features ({len(dropped_features)}):")
for f in dropped_features:
    print(f"  - {f}")

Total features before selection: 15

Retained features (7):
  + yrs_experience
  + income
  + mortgage_amt
  + fixed_deposit_acct
  + education_level_Advanced or Professional
  + education_level_Graduate
  + education_level_Undergraduate

Dropped features (8):
  - age
  - family_size
  - credit_card_spend
  - share_trading_acct
  - credit_card_acct_no
  - credit_card_acct_yes
  - online_acct_no
  - online_acct_yes


## Step 4: Chi2 Scores — Full Feature Ranking
Print the chi2 score for every feature to understand the statistical
justification for each selection and rejection decision.

In [6]:
scores_df = pd.DataFrame({
    'feature': X_train.columns,
    'chi2_score': selector.scores_,
    'selected': selected_mask
}).sort_values('chi2_score', ascending=False)

print("Full feature ranking by chi2 score:")
print(scores_df.to_string(index=False))

Full feature ranking by chi2 score:
                                 feature   chi2_score  selected
                                  income 33781.530793      True
                            mortgage_amt 11040.789646      True
                      fixed_deposit_acct   609.788634      True
           education_level_Undergraduate    96.117568      True
                          yrs_experience    80.867099      True
education_level_Advanced or Professional    34.703692      True
                education_level_Graduate    33.719750      True
                       credit_card_spend    33.264275     False
                             family_size    23.961543     False
                      share_trading_acct     2.259815     False
                                     age     0.836381     False
                    credit_card_acct_yes     0.535511     False
                     credit_card_acct_no     0.225203     False
                          online_acct_no     0.065359     False
    

## Step 5: Rebuild Named DataFrames from Headless Arrays
SelectKBest outputs raw NumPy arrays with no column headers.
Manually rebuild as named Pandas DataFrames using the identified
selected_features list so downstream notebooks can reference columns by name.

In [7]:
X_train_sel = pd.DataFrame(X_train_selected, columns=selected_features)
X_test_sel  = pd.DataFrame(X_test_selected,  columns=selected_features)

print(f"X_train_sel shape: {X_train_sel.shape}")
print(f"X_test_sel shape:  {X_test_sel.shape}")
print(f"\nColumns in selected set:\n{list(X_train_sel.columns)}")

X_train_sel shape: (4800, 7)
X_test_sel shape:  (1200, 7)

Columns in selected set:
['yrs_experience', 'income', 'mortgage_amt', 'fixed_deposit_acct', 'education_level_Advanced or Professional', 'education_level_Graduate', 'education_level_Undergraduate']


## Step 6: Export Selected Feature Sets
Save the selected train/test sets and the feature name list so
notebooks 05 and 06 can load them directly without repeating
the selection logic.

In [8]:
joblib.dump(X_train_sel,       '../data/X_train_sel.pkl')
joblib.dump(X_test_sel,        '../data/X_test_sel.pkl')
joblib.dump(y_train,           '../data/y_train.pkl')
joblib.dump(y_test,            '../data/y_test.pkl')
joblib.dump(selected_features, '../data/selected_features.pkl')

print("Exported:")
print("  ../data/X_train_sel.pkl")
print("  ../data/X_test_sel.pkl")
print("  ../data/y_train.pkl")
print("  ../data/y_test.pkl")
print("  ../data/selected_features.pkl")
print(f"\nSelected features for use in notebooks 05 & 06: {selected_features}")

Exported:
  ../data/X_train_sel.pkl
  ../data/X_test_sel.pkl
  ../data/y_train.pkl
  ../data/y_test.pkl
  ../data/selected_features.pkl

Selected features for use in notebooks 05 & 06: ['yrs_experience', 'income', 'mortgage_amt', 'fixed_deposit_acct', 'education_level_Advanced or Professional', 'education_level_Graduate', 'education_level_Undergraduate']
